In [3]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../6.Near Miss - 2016.xlsx"
sheet="Near Miss"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

original_shape=df.shape

print("Original Shape:", original_shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

replace_values=[
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan",
    "Not Mentioned",
    "not mentioned",
    "NOT MENTIONED"
]

df=df.replace(
    replace_values,
    np.nan
)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):
        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMNS
# ==========================

details_col=None
event_col=None

for col in df.columns:

    c=col.lower()

    if (
        "details" in c
        and
        "near" in c
    ):
        details_col=col

    if (
        "description" in c
        and
        "incident" in c
    ):
        event_col=col

# ==========================
# REMOVE ROW ONLY IF BOTH EMPTY
# ==========================

if details_col and event_col:

    before=len(df)

    keep=(
        (
            df[details_col]
            .fillna("")
            .str.strip()
            !=""
        )
        |
        (
            df[event_col]
            .fillna("")
            .str.strip()
            !=""
        )
    )

    df=df[keep]

    print(
        "\nRows Removed (Both description columns empty):",
        before-len(df)
    )

# ==========================
# FORMAT DATE COLUMNS
# ==========================

date_cols=[]

for col in df.columns:

    c=col.lower()

    if (
        "date" in c
        or
        "time_of_occurrence" in c
        or
        "date_&_time" in c
    ):
        date_cols.append(col)

def format_date(value):

    if pd.isna(value):
        return np.nan

    value=str(value).strip()

    if value.lower() in [
        "na",
        "n/a",
        "not mentioned",
        "nan"
    ]:
        return np.nan

    # already formatted → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        value
    ):
        return value

    try:

        dt=pd.to_datetime(
            value,
            errors="coerce"
        )

        if pd.notna(dt):

            return dt.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan

for col in date_cols:

    df[col]=df[col].apply(
        format_date
    )

print("\nFormatted Date Columns:")
print(date_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:", duplicates)

# ==========================
# RESET SERIAL NUMBER
# ==========================

si_col=None

for col in df.columns:

    c=(
        col.lower()
        .replace("_","")
        .replace(":","")
    )

    if (
        "slno" in c
        or
        "sino" in c
        or
        "serialno" in c
    ):
        si_col=col
        break

df=df.reset_index(
    drop=True
)

if si_col:

    df[si_col]=range(
        1,
        len(df)+1
    )

    print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_6_Near_Miss_2016.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:", output)

Original Shape: (805, 35)

Removed Empty Columns:
[]

Rows Removed (Both description columns empty): 320

Formatted Date Columns:
['Ref_num/Report_date', 'Date_and_Time_of_occurrence', 'Due_Date', 'Closure_Date_-_CA']

Duplicates Removed: 0

SI_No reordered

Missing Values Summary:
Ref_num/Report_date                             485
Port_Name                                       336
Details_of_near_miss                              3
Description_of_event_leading_to_the_incident    134
Immediate_action_taken                            1
Potential_extent_of_damage/injury               146
Cause_Analysis                                   48
Proposed_Corrective_Action                       62
Details_of_potential_loss_Category               30
Immediate_action_initiated                       28
Learning_Potential                               38
Severity_of_incident                             36
Corrective_Action                                54
PIC                                      